In [ ]:
# @title Загрузка Pascal VOC 2012

import os
import torch
import torchvision
from torchvision import datasets, models
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
import random
from torchvision.transforms import functional as F
from PIL import Image

print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚠️ Device: {device} (CPU режим)")


DATA_DIR = './data'
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 4
NUM_WORKERS = 0
NUM_EPOCHS = 3

class VOCSegmentationCustom(Dataset):
    def __init__(self, root, year, image_set, download=True):
        self.base_dataset = datasets.VOCSegmentation(
            root=root, year=year, image_set=image_set,
            download=download, transforms=None
        )
        self.is_train = (image_set == 'train')

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img_path = self.base_dataset.images[idx]
        mask_path = self.base_dataset.masks[idx]

        img = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path)

        img = F.resize(img, IMAGE_SIZE, interpolation=F.InterpolationMode.BILINEAR)
        mask = F.resize(mask, IMAGE_SIZE, interpolation=F.InterpolationMode.NEAREST)

        if self.is_train and random.random() > 0.5:
            img = F.hflip(img)
            mask = F.hflip(mask)

        img = F.to_tensor(img)
        img = F.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask = torch.as_tensor(np.array(mask), dtype=torch.int64)

        return img, mask

print(f"\nЗагрузка VOC ")

train_dataset = VOCSegmentationCustom(root=DATA_DIR, year='2012', image_set='train', download=True)
val_dataset = VOCSegmentationCustom(root=DATA_DIR, year='2012', image_set='val', download=True)

print(f"✓ Train: {len(train_dataset)} | Val: {len(val_dataset)}")

img, mask = train_dataset[0]
print(f"✓ Размер: {img.shape} (должно быть [3, 128, 128])")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

print("\n✓ Готово!")

PyTorch: 2.10.0+cpu
⚠️ Device: cpu (CPU режим)

Загрузка VOC 2012...
✓ Train: 1464 | Val: 1449
✓ Размер: torch.Size([3, 128, 128]) (должно быть [3, 128, 128])

✓ Готово!


In [ ]:
# @title Обучение DeepLabV3+ MobileNetV3

from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights

print("Загрузка лёгкой модели ...")
weights = DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
model = deeplabv3_mobilenet_v3_large(weights=weights)
model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

print(f"\n Начало обучения ({NUM_EPOCHS} эпох)...")
print("="*60)

import time
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    epoch_start = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / len(train_loader)
    scheduler.step()

    print(f"\n Epoch {epoch+1} завершена. Loss: {avg_loss:.4f} | Время: {epoch_time/60:.1f} мин")

total_time = (time.time() - start_time) / 3600
print(f"\n Обучение завершено! Общее время: {total_time:.2f} часов")

torch.save(model.state_dict(), 'model.pth')
print("✓ Модель сохранена")

Загрузка лёгкой модели (MobileNetV3)...
Downloading: "https://download.pytorch.org/models/deeplabv3_mobilenet_v3_large-fc3c493d.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_mobilenet_v3_large-fc3c493d.pth


100%|██████████| 42.3M/42.3M [00:00<00:00, 107MB/s]



🕐 Начало обучения (3 эпох)...
⏱️  Ожидаемое время: ~2-4 часа на CPU


Epoch 1/3: 100%|██████████| 366/366 [04:14<00:00,  1.44it/s, loss=0.4757]



✅ Epoch 1 завершена. Loss: 0.9440 | Время: 4.2 мин


Epoch 2/3: 100%|██████████| 366/366 [04:10<00:00,  1.46it/s, loss=0.8690]



✅ Epoch 2 завершена. Loss: 0.7002 | Время: 4.2 мин


Epoch 3/3: 100%|██████████| 366/366 [04:04<00:00,  1.50it/s, loss=0.4215]


✅ Epoch 3 завершена. Loss: 0.5818 | Время: 4.1 мин

🎉 Обучение завершено! Общее время: 0.21 часов
✓ Модель сохранена


In [ ]:
# @title Оценка и создание results.txt

def calculate_iou_per_class(pred, target):
    unique_classes = torch.unique(target)
    ious = []

    for cls in unique_classes:
        if cls == 255:
            continue
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union == 0:
            continue
        ious.append(intersection / union)

    return ious

random.seed(42)
all_indices = list(range(len(val_dataset)))
selected_indices = random.sample(all_indices, min(100, len(all_indices)))

print(f"Оценка на {len(selected_indices)} изображениях...")

results = []
all_mean_ious = []

model.eval()
with torch.no_grad():
    for idx in tqdm(selected_indices, desc="Evaluating"):
        img = Image.open(val_dataset.base_dataset.images[idx]).convert('RGB')
        mask = Image.open(val_dataset.base_dataset.masks[idx])
        img_name = os.path.basename(val_dataset.base_dataset.masks[idx])

        img = F.resize(img, IMAGE_SIZE, interpolation=F.InterpolationMode.BILINEAR)
        mask = F.resize(mask, IMAGE_SIZE, interpolation=F.InterpolationMode.NEAREST)

        img_tensor = F.to_tensor(img)
        img_tensor = F.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask_tensor = torch.as_tensor(np.array(mask), dtype=torch.int64)

        img_tensor = img_tensor.unsqueeze(0).to(device)
        output = model(img_tensor)['out']
        pred = output.argmax(dim=1).squeeze(0).cpu()

        ious = calculate_iou_per_class(pred, mask_tensor)

        mean_iou = sum(ious) / len(ious) if ious else 0.0
        num_objects = len(ious)

        all_mean_ious.append(mean_iou)
        results.append({'name': img_name, 'mean_iou': mean_iou, 'num_objects': num_objects, 'ious': ious})

global_mean_iou = sum(all_mean_ious) / len(all_mean_ious) if all_mean_ious else 0.0

print(f"\n✓ Средний IoU: {global_mean_iou:.6f}")

output_lines = [f"{global_mean_iou:.6f}"]
for res in results:
    ious_str = " ".join([f"{iou:.6f}" for iou in res['ious']])
    line = f"{res['name']} {res['mean_iou']:.6f} {res['num_objects']} {ious_str}"
    output_lines.append(line)

with open("results.txt", "w") as f:
    f.write("\n".join(output_lines))

print("✓ results.txt создан!")
print("\nПервые 5 строк:")
for line in output_lines[:5]:
    print(line[:100] + "..." if len(line) > 100 else line)

Оценка на 100 изображениях...


Evaluating: 100%|██████████| 100/100 [00:09<00:00, 11.05it/s]


✓ Средний IoU: 0.456529
✓ results.txt создан!

Первые 5 строк:
0.456529
2011_000953.png 0.545311 4 0.844728 0.000000 0.748768 0.587749
2007_007165.png 0.393738 2 0.787476 0.000000
2007_001587.png 0.490082 2 0.980164 0.000000
2008_006219.png 0.450410 2 0.900820 0.000000


In [ ]:
from google.colab import files
files.download('results.txt')
print("✓ Файл загружен!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Файл загружен!
